[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module4/06-sql-python.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module4/06-sql-python.ipynb)

# SQL and Python: sqlite3, SQLAlchemy, and Pandas Integration

**Module 4 — Data Science & Visualization** | Estimated time: 35 minutes

## Learning Objectives

By the end of this notebook you will be able to:
- Use `sqlite3` to create, populate, and query a relational database
- Write safe parameterized queries to prevent SQL injection
- Use context managers for clean connection handling
- Use SQLAlchemy's `create_engine` and `text()` for database access
- Read query results directly into Pandas DataFrames with `pd.read_sql()`
- Understand ORM basics and query optimization with indexes

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# SQLAlchemy comes pre-installed in Colab
from sqlalchemy import create_engine, text, inspect

print(f'SQLite version: {sqlite3.sqlite_version}')
print(f'Pandas  version: {pd.__version__}')

DB_PATH = '/tmp/inventory.db'
# Clean slate for reruns
Path(DB_PATH).unlink(missing_ok=True)
print(f'Database will be created at: {DB_PATH}')

## 1. sqlite3: Creating Tables and Inserting Data

The `sqlite3` module is part of the Python standard library. A **connection** links Python to the database file; a **cursor** executes SQL statements.

In [ ]:
# Use context manager — connection closes automatically on exit
with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    # Create tables
    cur.executescript("""
        CREATE TABLE IF NOT EXISTS suppliers (
            supplier_id   INTEGER PRIMARY KEY AUTOINCREMENT,
            name          TEXT NOT NULL,
            country       TEXT NOT NULL,
            lead_time_days INTEGER
        );

        CREATE TABLE IF NOT EXISTS categories (
            category_id   INTEGER PRIMARY KEY AUTOINCREMENT,
            name          TEXT NOT NULL UNIQUE
        );

        CREATE TABLE IF NOT EXISTS products (
            product_id    INTEGER PRIMARY KEY AUTOINCREMENT,
            sku           TEXT NOT NULL UNIQUE,
            name          TEXT NOT NULL,
            category_id   INTEGER REFERENCES categories(category_id),
            supplier_id   INTEGER REFERENCES suppliers(supplier_id),
            unit_price    REAL NOT NULL,
            stock_qty     INTEGER DEFAULT 0,
            reorder_level INTEGER DEFAULT 50
        );

        CREATE TABLE IF NOT EXISTS sales_orders (
            order_id      INTEGER PRIMARY KEY AUTOINCREMENT,
            product_id    INTEGER REFERENCES products(product_id),
            order_date    TEXT NOT NULL,
            quantity      INTEGER NOT NULL,
            unit_price    REAL NOT NULL,
            region        TEXT
        );
    """)
    conn.commit()
    print('Tables created: suppliers, categories, products, sales_orders')

    # Verify tables exist
    tables = cur.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
    print('Tables in DB:', [t[0] for t in tables])

## 2. Inserting Data with Parameterized Queries

Always use parameterized queries (`?` placeholders) instead of string formatting. String formatting allows **SQL injection** attacks where user-supplied input can modify or destroy your database.

In [ ]:
rng = np.random.default_rng(7)

suppliers_data = [
    ('TechParts Inc.', 'USA', 5),
    ('Global Components', 'China', 21),
    ('EuroParts GmbH', 'Germany', 14),
    ('SwiftSupply Co.', 'Canada', 7),
]

categories_data = [
    ('Electronics',), ('Accessories',), ('Software',), ('Hardware',)
]

products_data = [
    ('SKU-001', 'Wireless Mouse',      1, 1, 29.99, 200, 50),
    ('SKU-002', 'Mechanical Keyboard', 2, 2, 89.99, 120, 30),
    ('SKU-003', 'USB-C Hub',           1, 1, 49.99, 85,  25),
    ('SKU-004', 'Monitor Stand',       4, 3, 39.99, 60,  20),
    ('SKU-005', 'Webcam HD',           1, 2, 79.99, 150, 40),
    ('SKU-006', 'Laptop Bag',          2, 4, 59.99, 200, 50),
    ('SKU-007', 'Antivirus License',   3, 1, 19.99, 999, 100),
    ('SKU-008', 'SSD 1TB',             1, 2, 119.99, 75, 30),
    ('SKU-009', 'RAM 16GB',            1, 2, 69.99, 90,  25),
    ('SKU-010', 'Desk Lamp LED',       4, 4, 34.99, 180, 45),
]

# Generate orders
regions = ['North', 'South', 'East', 'West']
dates = pd.date_range('2024-01-01', '2024-12-31', freq='W').strftime('%Y-%m-%d').tolist()
orders = []
for _ in range(500):
    pid = int(rng.integers(1, 11))
    price = products_data[pid - 1][6]  # unit_price is index 6... let's use index 5
    price = products_data[pid - 1][5]   # Actually unit_price is at index 5 in tuple
    qty = int(rng.integers(1, 20))
    orders.append((pid, str(rng.choice(dates)), qty, price, str(rng.choice(regions))))

with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()
    # Use executemany for batch inserts — much faster than a loop
    cur.executemany('INSERT INTO suppliers (name, country, lead_time_days) VALUES (?,?,?)',
                    suppliers_data)
    cur.executemany('INSERT INTO categories (name) VALUES (?)', categories_data)
    cur.executemany(
        'INSERT INTO products (sku, name, category_id, supplier_id, unit_price, stock_qty, reorder_level) VALUES (?,?,?,?,?,?,?)',
        products_data
    )
    cur.executemany(
        'INSERT INTO sales_orders (product_id, order_date, quantity, unit_price, region) VALUES (?,?,?,?,?)',
        orders
    )
    conn.commit()
    print(f'Inserted: {len(suppliers_data)} suppliers, {len(categories_data)} categories, '
          f'{len(products_data)} products, {len(orders)} orders')

## 3. SELECT, WHERE, JOIN, and GROUP BY

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    conn.row_factory = sqlite3.Row  # access columns by name
    cur = conn.cursor()

    # Basic SELECT
    print('=== Products low on stock (below reorder level) ===')
    rows = cur.execute("""
        SELECT sku, name, stock_qty, reorder_level
        FROM products
        WHERE stock_qty < reorder_level
        ORDER BY stock_qty ASC
    """).fetchall()
    for r in rows:
        print(f"  {r['sku']}  {r['name']:<25} stock={r['stock_qty']:<5} reorder={r['reorder_level']}")

    # JOIN: products with their category and supplier
    print('\n=== Product catalog with category and supplier ===')
    rows = cur.execute("""
        SELECT p.sku, p.name, c.name AS category, s.name AS supplier, p.unit_price
        FROM products p
        JOIN categories c ON p.category_id = c.category_id
        JOIN suppliers  s ON p.supplier_id  = s.supplier_id
        ORDER BY p.unit_price DESC
    """).fetchall()
    for r in rows:
        print(f"  {r['sku']}  {r['name']:<25} {r['category']:<14} {r['supplier']:<22} ${r['unit_price']:.2f}")

    # GROUP BY: total revenue by product
    print('\n=== Top 5 products by total revenue ===')
    rows = cur.execute("""
        SELECT p.name, SUM(o.quantity) AS total_qty,
               SUM(o.quantity * o.unit_price) AS total_revenue
        FROM sales_orders o
        JOIN products p ON o.product_id = p.product_id
        GROUP BY p.product_id
        ORDER BY total_revenue DESC
        LIMIT 5
    """).fetchall()
    for r in rows:
        print(f"  {r['name']:<25} qty={r['total_qty']:<6} revenue=${r['total_revenue']:,.2f}")

## 4. Pandas + SQL: pd.read_sql() and to_sql()

`pd.read_sql()` executes a SQL query and returns a DataFrame directly. `df.to_sql()` writes a DataFrame to a database table.

In [ ]:
# Create SQLAlchemy engine (SQLite)
engine = create_engine(f'sqlite:///{DB_PATH}')

# Read into DataFrame
df_revenue = pd.read_sql("""
    SELECT
        o.region,
        strftime('%Y-%m', o.order_date) AS month,
        p.name AS product,
        c.name AS category,
        SUM(o.quantity * o.unit_price) AS revenue,
        SUM(o.quantity) AS units_sold
    FROM sales_orders o
    JOIN products   p ON o.product_id  = p.product_id
    JOIN categories c ON p.category_id = c.category_id
    GROUP BY o.region, month, p.product_id
    ORDER BY month, revenue DESC
""", con=engine)

print(f'Query returned {len(df_revenue)} rows')
print(df_revenue.head(10).to_string(index=False))

# Now use pandas for further analysis
print('\n=== Revenue by region ===')
print(df_revenue.groupby('region')['revenue'].sum().sort_values(ascending=False).apply('${:,.2f}'.format))

print('\n=== Revenue by category ===')
print(df_revenue.groupby('category')['revenue'].sum().sort_values(ascending=False).apply('${:,.2f}'.format))

# Write a summary table BACK to the database
df_summary = df_revenue.groupby(['region', 'category']).agg(
    total_revenue=('revenue', 'sum'),
    total_units=('units_sold', 'sum')
).reset_index()

df_summary.to_sql('revenue_summary', con=engine, if_exists='replace', index=False)
print('\nWrote revenue_summary table back to DB.')

# Verify
print(pd.read_sql('SELECT * FROM revenue_summary ORDER BY total_revenue DESC', engine).head(5).to_string(index=False))

## 5. Query Optimization: Indexes and EXPLAIN QUERY PLAN

In [ ]:
import time

with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    # EXPLAIN QUERY PLAN — without index
    print('=== EXPLAIN QUERY PLAN (no index on order_date) ===')
    plan = cur.execute("""
        EXPLAIN QUERY PLAN
        SELECT * FROM sales_orders WHERE order_date >= '2024-06-01'
    """).fetchall()
    for row in plan:
        print(' ', row)

    # Benchmark without index
    start = time.perf_counter()
    for _ in range(100):
        cur.execute("SELECT * FROM sales_orders WHERE order_date >= '2024-06-01'").fetchall()
    t_no_index = time.perf_counter() - start

    # Create index
    cur.execute('CREATE INDEX IF NOT EXISTS idx_order_date ON sales_orders(order_date)')
    conn.commit()
    print('\nIndex created on sales_orders(order_date)')

    # EXPLAIN after index
    print('\n=== EXPLAIN QUERY PLAN (with index) ===')
    plan = cur.execute("""
        EXPLAIN QUERY PLAN
        SELECT * FROM sales_orders WHERE order_date >= '2024-06-01'
    """).fetchall()
    for row in plan:
        print(' ', row)

    # Benchmark with index
    start = time.perf_counter()
    for _ in range(100):
        cur.execute("SELECT * FROM sales_orders WHERE order_date >= '2024-06-01'").fetchall()
    t_index = time.perf_counter() - start

print(f'\nTime without index: {t_no_index*1000:.1f} ms (100 queries)')
print(f'Time with    index: {t_index*1000:.1f} ms (100 queries)')
print(f'Speedup: {t_no_index/t_index:.1f}x')

## 6. ORM Basics with SQLAlchemy

The SQLAlchemy ORM lets you define database tables as Python classes and query them using Python syntax instead of raw SQL strings.

In [ ]:
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session
from sqlalchemy import String, Float, Integer

ORM_DB = 'sqlite:////tmp/orm_demo.db'
orm_engine = create_engine(ORM_DB, echo=False)

# Define the ORM model
class Base(DeclarativeBase):
    pass

class Product(Base):
    __tablename__ = 'products'
    id:         Mapped[int]   = mapped_column(Integer, primary_key=True)
    sku:        Mapped[str]   = mapped_column(String(20), unique=True)
    name:       Mapped[str]   = mapped_column(String(100))
    price:      Mapped[float] = mapped_column(Float)
    stock:      Mapped[int]   = mapped_column(Integer, default=0)

    def __repr__(self):
        return f'<Product {self.sku}: {self.name} ${self.price:.2f} stock={self.stock}>'

# Create tables
Base.metadata.create_all(orm_engine)

# Insert records using the session
with Session(orm_engine) as session:
    products = [
        Product(sku='A001', name='Alpha Widget',  price=19.99, stock=100),
        Product(sku='A002', name='Beta Gadget',   price=49.99, stock=50),
        Product(sku='A003', name='Gamma Gizmo',   price=34.99, stock=75),
    ]
    session.add_all(products)
    session.commit()
    print('ORM: inserted 3 products')

# Query using ORM
from sqlalchemy import select
with Session(orm_engine) as session:
    stmt = select(Product).where(Product.stock > 60).order_by(Product.price)
    results = session.scalars(stmt).all()
    print('\nProducts with stock > 60, sorted by price:')
    for p in results:
        print(' ', p)

## 7. Practical: Inventory Analytics Dashboard

In [ ]:
df_monthly = pd.read_sql("""
    SELECT
        strftime('%Y-%m', order_date) AS month,
        region,
        SUM(quantity * unit_price) AS revenue
    FROM sales_orders
    GROUP BY month, region
    ORDER BY month
""", con=engine)

df_pivot = df_monthly.pivot(index='month', columns='region', values='revenue').fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stacked area chart of monthly revenue by region
df_pivot.plot.area(ax=axes[0], alpha=0.7, colormap='tab10', linewidth=0)
axes[0].set_title('Monthly Revenue by Region')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Revenue ($)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(title='Region', loc='upper left')

# Bar chart: top products
df_top = pd.read_sql("""
    SELECT p.name, SUM(o.quantity * o.unit_price) AS total_revenue
    FROM sales_orders o JOIN products p ON o.product_id = p.product_id
    GROUP BY p.product_id ORDER BY total_revenue DESC LIMIT 10
""", con=engine)

axes[1].barh(df_top['name'], df_top['total_revenue'], color='steelblue', edgecolor='white')
axes[1].set_title('Top 10 Products by Revenue')
axes[1].set_xlabel('Total Revenue ($)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()
print('Inventory analytics dashboard rendered.')

## Practice Exercises

**Exercise 1 — Advanced SQL Query**
Write a SQL query that finds, for each supplier, the total number of orders placed, total revenue generated, and the percentage of total database revenue that each supplier accounts for. Use a subquery or CTE (`WITH ... AS`) for the total revenue.

**Exercise 2 — SQL Injection Prevention**
Demonstrate why parameterized queries are safer: write a function `get_product_by_name(conn, name)` that uses `?` placeholders. Then show what would happen if you naively used f-string formatting with the input `' OR 1=1 --`.

**Exercise 3 — Pandas to SQL Pipeline**
Generate a DataFrame of 50 hypothetical `returns` records (order_id, return_date, reason, refund_amount) and write it to the database using `to_sql`. Then write a SQL query that joins `returns` with `sales_orders` and `products` to find the top 3 products by refund amount.